# 고급통계학습 1차 과제

## Fashion MNIST 분류와 InceptionV3 전이학습

이 노트북은 `과제내용.md`와 제공된 세 참고 노트북을 바탕으로 작성하였다.

| 과제 | 실험 구성 | 확인할 결과 |
| --- | --- | --- |
| 클래스별 표본 시각화 | Fashion MNIST의 10개 클래스에서 각각 1장 선택 | 2행 5열 이미지 |
| DNN과 CNN 비교 | Fashion MNIST, 거의 동일한 모수 수와 동일한 학습 조건 | 학습 곡선, test accuracy, 정확도 차이 |
| 전이학습 | CIFAR-10 → ImageNet 사전학습 InceptionV3 → 다항 로지스틱 회귀 | CIFAR-10 CNN과의 test accuracy 비교 |

**실행 안내:** Python 3.11 등 설치한 TensorFlow 버전이 지원하는 환경 또는 Google Colab에서 위에서부터 모든 셀을 실행한다.
필요한 패키지는 `tensorflow`, `numpy`, `pandas`, `matplotlib`, `ipython`이다.
최초 실행에는 데이터와 ImageNet 가중치를 다운로드할 인터넷 연결이 필요하다. GPU 사용을 권장하며 CPU에서는 시간이 오래 걸릴 수 있다.

**현재 파일은 학습 실행 전 상태이다.** 정확도와 그래프는 실제 실행 결과로만 생성한다.
제출 전 모든 셀을 실행하고 출력이 포함되도록 노트북을 저장한다.


### 제공된 세 노트북과의 대응 및 보완 사항

| 참고 파일 | 이어받은 내용 | 이 노트북에서의 적용·보완 |
| --- | --- | --- |
| `DNN.ipynb` | 데이터 로드, Flatten·Dense·softmax, Adam과 교차엔트로피, 학습 곡선 | 원본의 MNIST 대신 과제에서 요구한 Fashion MNIST를 사용하고, 9장 임의 예시 대신 10개 클래스에서 각각 표본을 선택 |
| `DNN_vs_CNN.ipynb` | Fashion MNIST의 DNN/CNN 비교, 3개 합성곱층과 2개 풀링층, 20 epochs·batch 100 | DNN의 첫 은닉층을 110에서 104로 조정하여 98,460개였던 모수를 93,150개로 줄이고 CNN 93,322개와 더 가깝게 맞춤 |
| `Transfer_learning.ipynb` | CIFAR-10을 75×75로 확대, ImageNet InceptionV3 특징 2,048개, 단일 softmax 분류기 | 배치 단위 확대, 명시적 동결, `pooling='avg'`로 특징 차원 보장, 동일 CIFAR-10 테스트셋으로 직접 학습 CNN과 비교 |

원본 전이학습 코드의 75×75 입력에서는 마지막 특징 맵이 1×1×2048이므로, 공간 평균 풀링은 원본의 `squeeze`와 같은 특징 벡터를 만든다.
평균 풀링은 배치 크기가 1일 때도 배치 차원을 유지한다.
전처리는 이미지 모형에서는 0~1, InceptionV3에서는 전용 함수에 의한 −1~1을 사용한다. 추출된 특징은 다시 255로 나누지 않는다.

참고 코드의 `validation_split=0.10` 대신 고정한 클래스별 분할을 공유한다.
훈련 데이터는 먼저 무작위로 정한 인덱스로 배치하고, epoch마다 최대 10,000개 표본의 버퍼로 다시 섞어 메모리를 제한한다.
원본처럼 20회 모두 학습하되, 마지막 가중치 대신 최저 검증 손실의 가중치를 평가하는 변경을 명시하였다.
가중치 복원은 별도 콜백으로 수행하고 복원 후 검증 손실도 확인한다.
`model.summary()`로 구조를 확인하며 Graphviz가 필요한 `plot_model()`은 필수 의존성에서 제외하였다.


## 1. 실행 환경과 공통 함수

모든 분류 모형에 Adam(학습률 0.001), 20 epochs, batch size 100을 사용한다.
공식 훈련 자료의 10%를 클래스별로 나누어 검증 자료로 사용한다.
20회 학습을 모두 마친 뒤 검증 손실이 가장 낮았던 epoch의 가중치를 복원하고, 테스트 자료는 학습과 epoch 선택에 사용하지 않는다.
시드를 고정하되 실행 환경에 따른 작은 수치 차이는 발생할 수 있다.

참고 코드가 마지막 epoch의 가중치를 평가한 것과 달리, 여기서는 모든 모형에 동일한 검증 기반 선택 규칙을 적용한다. 따라서 참고 노트북에 저장된 정확도와 수치가 정확히 일치할 필요는 없다.


In [ ]:
# 필요한 패키지가 없다면 아래 줄의 주석을 해제하여 실행하고 커널을 재시작한다.
# %pip install tensorflow numpy pandas matplotlib ipython

import gc
import platform
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from IPython.display import display, Markdown
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input

SEED = 42
EPOCHS = 20
BATCH_SIZE = 100
FEATURE_BATCH_SIZE = 64
SHUFFLE_BUFFER_SIZE = 10000  # 특징 벡터 전체를 셔플 버퍼에 복제하지 않는다.
LEARNING_RATE = 1e-3
VALIDATION_RATIO = 0.10
INCEPTION_SIZE = 75  # 참고 코드와 동일한 입력 크기

keras.utils.set_random_seed(SEED)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': False})
print('Python:', platform.python_version())
print('TensorFlow:', tf.__version__)
print('NumPy:', np.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
print('모든 실험에서 전체 공식 훈련/테스트 자료를 사용합니다.')


In [ ]:
def stratified_indices(labels, validation_ratio=VALIDATION_RATIO, seed=SEED):
    """각 클래스의 비율을 유지하며 훈련/검증 인덱스를 한 번 결정한다."""
    labels = np.asarray(labels).reshape(-1)
    rng = np.random.default_rng(seed)
    train_parts, valid_parts = [], []
    for label in np.unique(labels):
        indices = rng.permutation(np.flatnonzero(labels == label))
        n_valid = int(round(len(indices) * validation_ratio))
        valid_parts.append(indices[:n_valid])
        train_parts.append(indices[n_valid:])
    train_idx = rng.permutation(np.concatenate(train_parts))
    valid_idx = rng.permutation(np.concatenate(valid_parts))
    assert np.intersect1d(train_idx, valid_idx).size == 0
    assert len(train_idx) + len(valid_idx) == len(labels)
    return train_idx, valid_idx


def make_dataset(inputs, labels, training=False):
    # 이미지와 이미 추출한 특징에 공통으로 사용한다. 여기서는 값을 변환하지 않는다.
    # 이미지의 정규화는 CNN/DNN의 Rescaling 층에서만 수행한다.
    assert len(inputs) == len(labels) and len(labels) > 0
    with tf.device('/CPU:0'):
        ds = tf.data.Dataset.from_tensor_slices((inputs, labels))
    if training:
        ds = ds.shuffle(min(len(labels), SHUFFLE_BUFFER_SIZE),
                        seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(1)



class BestValidationWeights(keras.callbacks.Callback):
    """학습을 조기 종료하지 않고 최저 검증 손실의 가중치를 복원한다."""
    def on_train_begin(self, logs=None):
        self.best_loss = float('inf')
        self.best_epoch = None
        self.best_weights = None

    def on_epoch_end(self, epoch, logs=None):
        value = (logs or {}).get('val_loss')
        if value is None or not np.isfinite(value):
            raise ValueError('유한한 val_loss가 필요합니다. 학습 결과를 확인하세요.')
        if float(value) < self.best_loss:
            self.best_loss = float(value)
            self.best_epoch = epoch + 1
            self.best_weights = self.model.get_weights()

    def on_train_end(self, logs=None):
        if self.best_weights is None:
            raise RuntimeError('복원할 검증 가중치가 없습니다.')
        self.model.set_weights(self.best_weights)
        self.best_weights = None  # 복원 후 보관용 메모리 해제


def fit_classifier(model, train_ds, valid_ds):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='sparse_categorical_crossentropy', metrics=['accuracy']
    )
    # 조기 종료와 가중치 선택을 혼용하지 않고 20회 학습 후 명시적으로 복원한다.
    selection = BestValidationWeights()
    start = perf_counter()
    history = model.fit(
        train_ds, validation_data=valid_ds, epochs=EPOCHS,
        callbacks=[selection], verbose=2
    )
    elapsed = perf_counter() - start
    best_epoch = selection.best_epoch
    assert len(history.history['loss']) == EPOCHS
    assert best_epoch == int(np.argmin(history.history['val_loss']) + 1)
    # 실제 복원된 가중치의 검증 손실과 기록상의 최솟값이 일치하는지 확인한다.
    restored = model.evaluate(valid_ds, verbose=0, return_dict=True)
    if not np.isclose(restored['loss'], selection.best_loss, rtol=1e-4, atol=1e-5):
        raise RuntimeError('복원한 모형의 검증 손실이 선택한 epoch와 다릅니다.')
    print(f'검증 손실 기준 선택 epoch: {best_epoch}, 학습 시간: {elapsed:.1f}초')
    return history, elapsed, best_epoch


def plot_histories(histories, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for name, history in histories.items():
        epochs = np.arange(1, len(history.history['loss']) + 1)
        for ax, metric in zip(axes, ['accuracy', 'loss']):
            line, = ax.plot(epochs, history.history[metric], '--', label=f'{name}: train')
            ax.plot(epochs, history.history[f'val_{metric}'],
                    color=line.get_color(), label=f'{name}: validation')
            ax.set(xlabel='Epoch', ylabel=metric.capitalize())
            ax.legend(fontsize=8)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def result_row(dataset, name, model, test_ds, seconds, best_epoch, frozen_params=0):
    scores = model.evaluate(test_ds, verbose=0, return_dict=True)
    trainable = sum(int(np.prod(w.shape)) for w in model.trainable_weights)
    return {
        'dataset': dataset, 'model': name,
        'trainable_params': trainable,
        'total_params': model.count_params() + frozen_params,
        'best_epoch': best_epoch,
        'test_samples': int(sum(int(tf.shape(y)[0]) for _, y in test_ds)),
        'test_loss': float(scores['loss']),
        'test_accuracy': float(scores['accuracy']), 'fit_seconds': seconds,
    }


## 2. Fashion MNIST: 클래스별 표본 그리기

Fashion MNIST는 28×28 흑백 의류 이미지로 구성된다. 공식 훈련 자료 60,000장과 테스트 자료 10,000장을 불러온다.
무작위 10장을 고르면 일부 클래스가 빠질 수 있으므로, **각 클래스에 해당하는 인덱스에서 1장씩** 선택한다.
클래스 이름과 레이블은 [Keras Fashion MNIST 문서](https://keras.io/api/datasets/fashion_mnist/#load_data-function)를 따른다.


In [ ]:
(fashion_x, fashion_y), (fashion_test_x, fashion_test_y) = keras.datasets.fashion_mnist.load_data()
fashion_names = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]
assert fashion_x.shape == (60000, 28, 28)
assert fashion_test_x.shape == (10000, 28, 28)
print('공식 훈련 자료:', fashion_x.shape, '테스트 자료:', fashion_test_x.shape)

rng = np.random.default_rng(SEED)
sample_indices = [int(rng.choice(np.flatnonzero(fashion_y == label))) for label in range(10)]
assert np.array_equal(fashion_y[sample_indices], np.arange(10))
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for label, (ax, index) in enumerate(zip(axes.flat, sample_indices)):
    ax.imshow(fashion_x[index], cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'{label}: {fashion_names[label]}')
    ax.axis('off')
fig.suptitle('Fashion MNIST: one sample per class')
plt.tight_layout()
plt.show()


## 3. Fashion MNIST: 비슷한 모수 수의 DNN과 CNN 비교

### 3.1 자료 분할과 모형 구조

DNN은 이미지를 784차원 벡터로 펼쳐 완전연결층에 전달한다.
CNN은 작은 필터의 가중치를 공간 위치마다 공유하여 국소적인 이미지 패턴을 학습한다.
두 모형은 동일한 훈련 54,000장, 검증 6,000장, 테스트 10,000장을 사용한다.
모두 입력을 255로 나누고, 데이터 증강과 dropout 없이 구조 차이를 비교한다.

| 모형 | 구조 | 모수 수 |
| --- | --- | ---: |
| DNN | Flatten → Dense(104) → Dense(100) → Dense(10) | 93,150 |
| CNN | Conv(32) → Pool → Conv(64) → Pool → Conv(64) → Flatten → Dense(64) → Dense(10) | 93,322 |

합성곱은 모두 3×3, `padding='valid'`, 풀링은 2×2이다.
은닉층에는 ReLU, 출력층에는 softmax를 적용한다.
모수 차이는 172개로 CNN 기준 약 0.18%이다.
Dense의 모수 수는 `(입력 수 + 1) × 출력 수`, Conv2D의 모수 수는 `(커널 높이 × 커널 너비 × 입력 채널 수 + 1) × 필터 수`로 계산한다.


In [ ]:
fashion_train_idx, fashion_valid_idx = stratified_indices(fashion_y)
fashion_images = fashion_x[..., np.newaxis]
fashion_valid_ds = make_dataset(
    fashion_images[fashion_valid_idx], fashion_y[fashion_valid_idx])
fashion_test_ds = make_dataset(fashion_test_x[..., np.newaxis], fashion_test_y)

display(pd.DataFrame({
    'class': fashion_names,
    'train': np.bincount(fashion_y[fashion_train_idx], minlength=10),
    'validation': np.bincount(fashion_y[fashion_valid_idx], minlength=10),
    'test': np.bincount(fashion_test_y, minlength=10),
}))


def build_dnn():
    return keras.Sequential([
        keras.Input(shape=(28, 28, 1)),
        layers.Rescaling(1.0 / 255),
        layers.Flatten(),
        layers.Dense(104, activation='relu'),
        layers.Dense(100, activation='relu'),
        layers.Dense(10, activation='softmax'),
    ], name='fashion_dnn')


def build_cnn(input_shape, name):
    return keras.Sequential([
        keras.Input(shape=input_shape),
        layers.Rescaling(1.0 / 255),
        layers.Conv2D(32, 3, activation='relu'),
        layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, activation='relu'),
        layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, activation='relu'),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax'),
    ], name=name)


keras.utils.set_random_seed(SEED)
fashion_dnn = build_dnn()
keras.utils.set_random_seed(SEED)
fashion_cnn = build_cnn((28, 28, 1), 'fashion_cnn')
assert fashion_dnn.count_params() == 93150
assert fashion_cnn.count_params() == 93322
param_gap = abs(fashion_dnn.count_params() - fashion_cnn.count_params()) / fashion_cnn.count_params()
assert param_gap < 0.01
fashion_dnn.summary()
fashion_cnn.summary()
print(f'모수 수 차이: {param_gap:.3%}')


### 3.2 학습과 테스트 평가

각 모형을 학습하기 직전에 시드를 다시 고정하고 동일한 방식으로 훈련 자료를 섞는다.
아래 정확도는 학습 자료에 대한 정확도가 아니라 별도의 공식 테스트 자료에 대한 정확도이다.


In [ ]:
keras.utils.set_random_seed(SEED)
fashion_train_ds = make_dataset(
    fashion_images[fashion_train_idx], fashion_y[fashion_train_idx], training=True)
dnn_history, dnn_seconds, dnn_epoch = fit_classifier(
    fashion_dnn, fashion_train_ds, fashion_valid_ds)

keras.utils.set_random_seed(SEED)
fashion_train_ds = make_dataset(
    fashion_images[fashion_train_idx], fashion_y[fashion_train_idx], training=True)
cnn_history, cnn_seconds, cnn_epoch = fit_classifier(
    fashion_cnn, fashion_train_ds, fashion_valid_ds)

plot_histories({'DNN': dnn_history, 'CNN': cnn_history}, 'Fashion MNIST: DNN vs CNN')
fashion_rows = [
    result_row('Fashion MNIST', 'DNN', fashion_dnn, fashion_test_ds, dnn_seconds, dnn_epoch),
    result_row('Fashion MNIST', 'CNN', fashion_cnn, fashion_test_ds, cnn_seconds, cnn_epoch),
]
fashion_results = pd.DataFrame(fashion_rows)
display(fashion_results.round(4))
fashion_gap_pp = 100 * (fashion_rows[1]['test_accuracy'] - fashion_rows[0]['test_accuracy'])
display(Markdown(
    f"**Fashion MNIST 결과:** DNN 정확도는 {fashion_rows[0]['test_accuracy']:.2%}, "
    f"CNN 정확도는 {fashion_rows[1]['test_accuracy']:.2%}이다. "
    f"CNN − DNN 정확도 차이는 **{fashion_gap_pp:+.2f}%p**이다."
))


### 3.3 해석 기준

CNN의 테스트 정확도가 높으면, 거의 동일한 모수 수에서도 국소 연결과 가중치 공유가 이 실험의 이미지 분류에 유리했다고 해석할 수 있다.
반대로 차이가 작거나 CNN이 낮다면 이 설정에서의 결과를 그대로 보고한다.
학습 정확도는 계속 증가하지만 검증 손실이 커지면 과적합의 신호이다.
모수 수만 맞췄다고 연산량이나 최적화 난이도까지 같아지는 것은 아니며, 한 시드의 실험으로 모든 CNN이 항상 우수하다고 일반화할 수는 없다.


## 4. CIFAR-10: 전이학습의 비교 기준이 되는 CNN

전이학습 과제에서 사용하는 자료는 CIFAR-10이다.
**Fashion MNIST의 CNN 정확도와 CIFAR-10의 전이학습 정확도를 직접 비교하면 데이터 난이도 차이가 섞인다.**
따라서 앞 절과 같은 합성곱 구조를 CIFAR-10 입력 크기에 맞춰 다시 학습한 뒤, 같은 테스트 이미지에서 전이학습과 비교한다.
훈련 45,000장과 검증 5,000장의 인덱스를 한 번 정하고 두 방법이 공유한다.


In [ ]:
# 앞 실험의 결과표와 표시한 그래프를 남기고 이미지/모형/history 메모리를 정리한다.
del fashion_dnn, fashion_cnn, fashion_train_ds, fashion_valid_ds, fashion_test_ds
del dnn_history, cnn_history
del fashion_x, fashion_y, fashion_test_x, fashion_test_y, fashion_images
keras.backend.clear_session()
gc.collect()

(cifar_x, cifar_y), (cifar_test_x, cifar_test_y) = keras.datasets.cifar10.load_data()
cifar_y = cifar_y.reshape(-1)
cifar_test_y = cifar_test_y.reshape(-1)
cifar_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']
assert cifar_x.shape == (50000, 32, 32, 3)
assert cifar_test_x.shape == (10000, 32, 32, 3)
cifar_train_idx, cifar_valid_idx = stratified_indices(cifar_y)
display(pd.DataFrame({
    'class': cifar_names,
    'train': np.bincount(cifar_y[cifar_train_idx], minlength=10),
    'validation': np.bincount(cifar_y[cifar_valid_idx], minlength=10),
    'test': np.bincount(cifar_test_y, minlength=10),
}))


# 참고 전이학습 노트북처럼 CIFAR-10 원본 이미지도 확인한다.
rng = np.random.default_rng(SEED)
cifar_sample_indices = [int(rng.choice(np.flatnonzero(cifar_y == k))) for k in range(10)]
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for k, (ax, index) in enumerate(zip(axes.flat, cifar_sample_indices)):
    ax.imshow(cifar_x[index])
    ax.set_title(f'{k}: {cifar_names[k]}')
    ax.axis('off')
fig.suptitle('CIFAR-10: one sample per class')
plt.tight_layout()
plt.show()

keras.utils.set_random_seed(SEED)
cifar_cnn = build_cnn((32, 32, 3), 'cifar_cnn')
assert cifar_cnn.count_params() == 122570
cifar_cnn.summary()
cifar_train_ds = make_dataset(cifar_x[cifar_train_idx], cifar_y[cifar_train_idx], training=True)
cifar_valid_ds = make_dataset(cifar_x[cifar_valid_idx], cifar_y[cifar_valid_idx])
cifar_test_ds = make_dataset(cifar_test_x, cifar_test_y)
cifar_history, cifar_seconds, cifar_epoch = fit_classifier(
    cifar_cnn, cifar_train_ds, cifar_valid_ds)
cifar_cnn_row = result_row(
    'CIFAR-10', 'CNN from scratch', cifar_cnn, cifar_test_ds, cifar_seconds, cifar_epoch)
display(pd.DataFrame([cifar_cnn_row]).round(4))


## 5. InceptionV3 특징 벡터와 다항 로지스틱 회귀

### 5.1 특징 추출

[Keras InceptionV3 문서](https://keras.io/api/applications/inceptionv3/inception_v3_model/)에 따라 `weights='imagenet'`, `include_top=False`로 불러온다.
`pooling='avg'`를 사용하여 이미지마다 2,048차원 특징 벡터를 얻는다.
참고 코드와 같이 입력 이미지를 75×75로 확대한다. 이는 허용되는 최소 크기이며, 일반적인 299×299 입력과 성능·연산량이 다를 수 있다.

[전용 전처리 함수](https://keras.io/api/applications/inceptionv3/inception_v3_preprocessing/)에는 **0~255 범위**의 실수 픽셀을 전달한다.
이 함수가 −1~1로 변환하므로 미리 255로 나누지 않는다.
이미지 전체를 확대하여 저장하지 않고 배치별로 확대해 메모리 사용을 줄인다.
특징 추출 시 순서를 섞지 않아 원래 레이블과의 대응을 유지한다.

InceptionV3의 가중치를 동결하고 추론 모드로 특징을 추출하므로 CIFAR-10의 훈련·검증·테스트 자료로 사전학습 모형을 갱신하지 않는다.


In [ ]:
del cifar_cnn, cifar_train_ds, cifar_valid_ds, cifar_test_ds
# history.model의 참조를 제거하되 학습 곡선은 보존한다.
cifar_history_values = dict(cifar_history.history)
del cifar_history
keras.backend.clear_session()
gc.collect()

feature_extractor = InceptionV3(
    weights='imagenet', include_top=False,
    input_shape=(INCEPTION_SIZE, INCEPTION_SIZE, 3), pooling='avg'
)
feature_extractor.trainable = False
backbone_params = feature_extractor.count_params()
assert len(feature_extractor.trainable_weights) == 0
print('InceptionV3 전체 모수 수(동결):', f'{backbone_params:,}')
print('특징 출력 형태:', feature_extractor.output_shape)


def feature_dataset(images):
    with tf.device('/CPU:0'):
        ds = tf.data.Dataset.from_tensor_slices(images).batch(FEATURE_BATCH_SIZE)
    def prepare(batch):
        resized = tf.image.resize(tf.cast(batch, tf.float32),
                                  (INCEPTION_SIZE, INCEPTION_SIZE))
        return preprocess_input(resized)
    return ds.map(prepare, num_parallel_calls=2, deterministic=True).prefetch(1)


start = perf_counter()
# predict는 training=False로 실행되어 BatchNormalization 통계를 변경하지 않는다.
cifar_features = feature_extractor.predict(feature_dataset(cifar_x), verbose=1)
cifar_test_features = feature_extractor.predict(feature_dataset(cifar_test_x), verbose=1)
feature_seconds = perf_counter() - start
assert cifar_features.shape == (50000, 2048)
assert cifar_test_features.shape == (10000, 2048)
assert np.isfinite(cifar_features).all() and np.isfinite(cifar_test_features).all()
print('전체 훈련 자료 특징:', cifar_features.shape)
print('테스트 특징:', cifar_test_features.shape)
print(f'훈련/검증/테스트 특징 추출 시간: {feature_seconds:.1f}초')

del feature_extractor, cifar_x, cifar_test_x
keras.backend.clear_session()
gc.collect()


### 5.2 다항 로지스틱 회귀 학습

특징 벡터 $z \in \mathbb{R}^{2048}$에 대해 다음 확률을 학습한다.

$$
P(Y=k \mid z)=
\frac{\exp(w_k^\top z+b_k)}{\sum_{j=0}^{9}\exp(w_j^\top z+b_j)}.
$$

은닉층 없이 `Dense(10, activation='softmax')` 한 층만 사용하면 다항 로지스틱 회귀가 된다.
교차엔트로피를 최소화하며, 학습할 모수는 $2048\times10+10=20,490$개이다.
이는 **분류기만의 학습 모수 수**이며, 실제 예측에는 동결된 InceptionV3도 필요하다.


In [ ]:
keras.utils.set_random_seed(SEED)
logistic_model = keras.Sequential([
    keras.Input(shape=(2048,)),
    layers.Dense(10, activation='softmax'),
], name='multinomial_logistic_regression')
assert logistic_model.count_params() == 20490
logistic_model.summary()

# 원본 이미지와 동일한 인덱스로 특징을 분리한다.
feature_train_ds = make_dataset(
    cifar_features[cifar_train_idx], cifar_y[cifar_train_idx], training=True)
feature_valid_ds = make_dataset(
    cifar_features[cifar_valid_idx], cifar_y[cifar_valid_idx])
feature_test_ds = make_dataset(cifar_test_features, cifar_test_y)
logistic_history, logistic_seconds, logistic_epoch = fit_classifier(
    logistic_model, feature_train_ds, feature_valid_ds)
transfer_row = result_row(
    'CIFAR-10', 'InceptionV3 + logistic', logistic_model, feature_test_ds,
    logistic_seconds, logistic_epoch, frozen_params=backbone_params)
display(pd.DataFrame([transfer_row]).round(4))


## 6. 비교 결과와 결론

표의 `trainable_params`는 이번 학습에서 갱신하는 모수 수이고, `total_params`는 동결된 모수까지 포함한다.
`fit_seconds`는 분류기 학습 시간이다. 전이학습의 특징 추출 시간은 따로 표시한다.
특징 추출 시간에는 훈련·검증·테스트 자료 처리가 포함되지만 다운로드 시간과 ImageNet 사전학습 시간은 포함되지 않는다.
따라서 분류기 학습 시간만으로 두 방법의 전체 비용을 비교하지 않는다.


In [ ]:
# plot_histories에 전달하기 위한 간단한 history 컨테이너
from types import SimpleNamespace
plot_histories({
    'CNN from scratch': SimpleNamespace(history=cifar_history_values),
    'InceptionV3 + logistic': logistic_history,
}, 'CIFAR-10: CNN vs transfer learning')

cifar_rows = [cifar_cnn_row, transfer_row]
all_results = pd.DataFrame(fashion_rows + cifar_rows)
display(all_results.round(4))
print(f'InceptionV3 특징 추출 시간: {feature_seconds:.1f}초')
print(f'전이학습 특징 추출 + 분류기 학습 시간: {feature_seconds + logistic_seconds:.1f}초')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, title, rows in zip(axes, ['Fashion MNIST', 'CIFAR-10'], [fashion_rows, cifar_rows]):
    values = [row['test_accuracy'] for row in rows]
    bars = ax.bar([row['model'] for row in rows], values, color=['#4776b4', '#d07b32'])
    ax.set(title=title, ylabel='Test accuracy', ylim=(0, 1))
    ax.bar_label(bars, labels=[f'{value:.2%}' for value in values], padding=3)
plt.tight_layout()
plt.show()

transfer_gap_pp = 100 * (transfer_row['test_accuracy'] - cifar_cnn_row['test_accuracy'])
fashion_judgment = (
    '거의 동일한 모수 수에서 CNN의 정확도가 더 높게 나타났다.' if fashion_gap_pp > 0
    else '이 설정에서는 CNN의 정확도가 DNN보다 높지 않았다.'
)
transfer_judgment = (
    '이 설정에서는 사전학습 특징을 사용한 전이학습의 정확도가 더 높았다.' if transfer_gap_pp > 0
    else '이 설정에서는 전이학습의 정확도가 직접 학습한 CNN보다 높지 않았다.'
)
display(Markdown(f'''
### 실제 실행 결과 요약

1. **클래스별 표본:** Fashion MNIST의 레이블 0~9에서 각각 1장을 선택하여 10개 클래스를 시각화하였다.
2. **DNN과 CNN:** 테스트 정확도는 DNN **{fashion_rows[0]['test_accuracy']:.2%}**, CNN **{fashion_rows[1]['test_accuracy']:.2%}**이며, CNN − DNN 차이는 **{fashion_gap_pp:+.2f}%p**이다. {fashion_judgment}
3. **전이학습:** CIFAR-10 CNN의 테스트 정확도는 **{cifar_cnn_row['test_accuracy']:.2%}**, InceptionV3 + 다항 로지스틱 회귀는 **{transfer_row['test_accuracy']:.2%}**이며, 전이학습 − CNN 차이는 **{transfer_gap_pp:+.2f}%p**이다. {transfer_judgment}
'''))


### 해석과 실험의 한계

- **DNN과 CNN의 차이:** CNN은 이미지의 국소적 공간 구조를 활용하고 필터를 공유한다. 모수 수가 비슷해도 이러한 구조적 가정과 최적화 특성에 따라 일반화 성능이 달라진다.
- **전이학습의 의미:** ImageNet에서 학습한 특징을 재사용하면 CIFAR-10에서는 선형 분류기만 학습할 수 있다. 다만 ImageNet이라는 추가 자료와 큰 사전학습 모형을 사용하므로, CIFAR-10만으로 처음부터 학습한 작은 CNN과 동일한 정보·모수 예산을 가진 실험은 아니다.
- **입력 크기와 도메인:** CIFAR-10 원본은 32×32이고, InceptionV3에는 이를 75×75로 확대하여 입력했다. 확대가 새로운 정보를 만들지는 않으며, ImageNet과 CIFAR-10의 영상 특성 차이도 성능에 영향을 줄 수 있다.
- **검증과 재현성:** 테스트 정확도를 보고 모형이나 epoch를 다시 선택하지 않는다. 본 비교는 한 번의 분할과 시드에 따른 관측이며, 신뢰성 있는 일반화를 위해서는 여러 시드의 평균과 표준편차를 추가로 확인할 수 있다.

## 참고 자료

- 과제 지시문: 같은 폴더의 `과제내용.md`
- 제공 코드: `DNN.ipynb`, `DNN_vs_CNN.ipynb`, `Transfer_learning.ipynb`
- [Keras Fashion MNIST](https://keras.io/api/datasets/fashion_mnist/#load_data-function)
- [Keras InceptionV3 모형](https://keras.io/api/applications/inceptionv3/inception_v3_model/)
- [Keras InceptionV3 전처리](https://keras.io/api/applications/inceptionv3/inception_v3_preprocessing/)

- [Keras 사용자 정의 콜백](https://keras.io/guides/writing_your_own_callbacks/)
